In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import SparkSession

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS control")
spark.sql("CREATE SCHEMA IF NOT EXISTS raw")
spark.sql("CREATE SCHEMA IF NOT EXISTS presentation")

spark.sql("""
CREATE TABLE IF NOT EXISTS control.sources (
    source_id        STRING,
    country_code     STRING,
    source_name      STRING,
    endpoint_url     STRING,
    format           STRING,
    parser           STRING,
    auth_type        STRING,
    secret_name      STRING,
    source_timezone  STRING,
    target_table     STRING,
    lookback_days    INT,
    window_size_days INT,
    start_date       DATE,
    active           BOOLEAN,
    updated_at_utc   TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS control.run_log (
    run_id           STRING,
    source_id        STRING,
    window_start_utc TIMESTAMP,
    window_end_utc   TIMESTAMP,
    status           STRING,
    rows_loaded      INT,
    error_message    STRING,
    executed_at_utc  TIMESTAMP
) USING DELTA
""")

In [ ]:
df = spark.sql("""
    SELECT * FROM raw.prices_es
    UNION ALL SELECT * FROM raw.prices_ro
    UNION ALL SELECT * FROM raw.prices_de
    UNION ALL SELECT * FROM raw.prices_pl
""").toPandas()

notebookutils.fs.put("Files/data.csv", df.to_csv(index=False), True)